<a href="https://colab.research.google.com/github/lucaslezcano03/Proyecto-IA/blob/main/dataset_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Sección 1: Instalación e Importación**
Configuración del entorno y librerías necesarias.

In [ ]:
# Instalación silenciosa de dependencias
!pip install langchain langchain-community --quiet

import pandas as pd
import langchain

print("✓ Librerías importadas correctamente.")

✓ Librerías importadas correctamente.


### **Sección 2: Carga del Dataset**
Carga inicial del archivo CSV con codificación compatible.

In [ ]:
archivo_original = 'sales_data_sample.csv'

try:
    df = pd.read_csv(archivo_original, encoding='unicode_escape')
    print(f"✓ Dataset original '{archivo_original}' cargado con éxito.")
    print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
except FileNotFoundError:
    print("❌ Error: Sube 'sales_data_sample.csv' a la barra lateral de Colab.")

✓ Dataset original 'sales_data_sample.csv' cargado con éxito.
Dimensiones: 2823 filas y 25 columnas.


### **Sección 3: Normalización y Limpieza**
Tratamiento de valores nulos y conversión de tipos de datos.

In [ ]:
if 'df' in locals():
    # 1. Normalizar columnas de tipo Objeto (Texto)
    cols_object = df.select_dtypes(include=['object']).columns
    df[cols_object] = df[cols_object].fillna('N/A')

    # 2. Normalizar columnas numéricas (int/float)
    cols_numeric = df.select_dtypes(include=['number']).columns
    df[cols_numeric] = df[cols_numeric].fillna(0)

    # 3. Estandarización de fechas
    if 'ORDERDATE' in df.columns:
        df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])

    # VALIDACIÓN: Verificar si quedan nulos
    nulos_finales = df.isnull().sum().sum()
    print(f"✓ Proceso de normalización completado.")
    print(f"✓ Total de valores nulos en el dataset: {nulos_finales}")
    if nulos_finales == 0:
        print("✅ Validación exitosa: El dataset está 100% limpio.")

✓ Proceso de normalización completado.
✓ Total de valores nulos en el dataset: 0
✅ Validación exitosa: El dataset está 100% limpio.


### **Sección 4: Exportación y Verificación**
Guardado del archivo limpio y visualización de resultados.

In [ ]:
if 'df' in locals():
    # Nuevo nombre solicitado
    nombre_nuevo_archivo = 'ventas.csv'

    # Exportación
    df.to_csv(nombre_nuevo_archivo, index=False)
    print(f"✓ Archivo creado exitosamente como: {nombre_nuevo_archivo}")

    # Verificación de las primeras filas
    display(df.head(3))

✓ Archivo creado exitosamente como: ventas.csv


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium


# Sección 5: Inicialización del Agente Inteligente
# Configuración del modelo Mistral Small con capacidad de ejecución de código.

In [14]:
### **Sección 5: Inicialización del Agente Inteligente**
# Configuración del modelo Mistral Small con capacidad de ejecución de código.

!pip install -q langchain langchain-mistralai pandas langchain_experimental

from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
import pandas as pd

# 1. Configuración de API y Modelo
api_key = userdata.get('MISTRAL_API_KEY')
llm = ChatMistralAI(api_key=api_key, model="mistral-small-latest")

# 2. Cargar el corpus normalizado
df = pd.read_csv('ventas.csv')

# 3. Prompt de precisión con contexto detallado del dataset
prefix_prompt = """
Eres un analista de datos experto en ventas. Tienes acceso a un dataframe de Pandas llamado 'df'.

Contexto de las columnas principales:
- ORDERNUMBER: Número único de identificación de la orden.
- QUANTITYORDERED: Cantidad de productos pedidos en esa línea.
- PRICEEACH: Precio de cada unidad del producto.
- SALES: Monto total de la venta (generalmente QUANTITYORDERED * PRICEEACH).
- ORDERDATE: Fecha en la que se realizó el pedido.
- STATUS: Estado actual (ej. Shipped, Cancelled, Resolved, On Hold).
- PRODUCTLINE: Categoría o línea del producto (ej. Motorcycles, Classic Cars).
- CUSTOMERNAME: Nombre del cliente o empresa.
- CITY y COUNTRY: Ubicación geográfica del cliente.

Instrucciones críticas:
1. Para cualquier pregunta numérica o estadística, DEBES generar y ejecutar código Python.
2. Si te preguntan por ventas totales, usa la columna 'SALES'.
3. Si hay valores 'N/A', ignóralos en cálculos matemáticos para no sesgar los resultados.
4. Responde siempre en español de forma clara y concisa.
"""

# 4. Creación del agente
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix_prompt,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling"
)

print("✓ Agente configurado con contexto de negocio.")

✓ Agente configurado con contexto de negocio.


In [15]:
# Prueba de funcionamiento
pregunta = "¿cuantas filas tiene el dataset?"
respuesta = agent.invoke(pregunta)
print(f"Respuesta del Agente: {respuesta['output']}")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'len(df)'}`


2823El dataset tiene **2,823 filas**.

> Finished chain.
Respuesta del Agente: El dataset tiene **2,823 filas**.


In [16]:
preguntas_prueba = [
    "¿Cuál fue el total de ventas (SALES) por cada año disponible?",
    "¿Cuáles son los 3 países que generaron más ingresos totales?",
    "¿Cuántos pedidos tienen un STATUS de 'Cancelled'?",
    "¿Cuál es el promedio de PRICEEACH para la línea 'Classic Cars'?"
]

for i, pregunta in enumerate(preguntas_prueba, 1):
    print(f"\n--- PRUEBA {i}: {pregunta} ---")
    try:
        resultado = agent.invoke(pregunta)
        print(f"\nRESPUESTA: {resultado['output']}")
    except Exception as e:
        print(f"Error en la prueba {i}: {str(e)}")


--- PRUEBA 1: ¿Cuál fue el total de ventas (SALES) por cada año disponible? ---


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Calcular el total de ventas (SALES) por cada año disponible en el dataframe\ntotal_ventas_por_año = df.groupby('YEAR_ID')['SALES'].sum().reset_index()\ntotal_ventas_por_año"}`


   YEAR_ID       SALES
0     2003  3516979.54
1     2004  4724162.60
2     2005  1791486.71El total de ventas (**SALES**) por cada año disponible es el siguiente:

| **Año** | **Total de Ventas** |
|---------|---------------------|
| 2003    | 3,516,979.54        |
| 2004    | 4,724,162.60        |
| 2005    | 1,791,486.71        |

> Finished chain.

RESPUESTA: El total de ventas (**SALES**) por cada año disponible es el siguiente:

| **Año** | **Total de Ventas** |
|---------|---------------------|
| 2003    | 3,516,979.54        |
| 2004    | 4,724,162.60        |
| 2005    | 1,791,486.71        |

--- PRUEBA 2: ¿Cuáles son los 3 países que g

### **Sección 6: Implementación de RAG Vectorial con ChromaDB**
En esta sección crearemos la base de conocimientos vectorial para búsquedas semánticas.

In [ ]:
# 1. Instalación de librerías para RAG y Vectores
!pip install -q chromadb sentence-transformers langchain-huggingface

import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 2. Preparación de Documentos: Convertir cada fila del CSV en un Documento
df_ventas = pd.read_csv('ventas.csv')

documentos = []
for _, row in df_ventas.iterrows():
    contenido = f"""
    Orden: {row['ORDERNUMBER']} | Cliente: {row['CUSTOMERNAME']}
    Producto: {row['PRODUCTLINE']} | Cantidad: {row['QUANTITYORDERED']}
    Precio Unit: {row['PRICEEACH']} | Venta Total: {row['SALES']}
    Fecha: {row['ORDERDATE']} | Estado: {row['STATUS']}
    Ubicación: {row['CITY']}, {row['COUNTRY']}
    """.strip()

    doc = Document(
        page_content=contenido,
        metadata={"source": "ventas.csv", "order": row['ORDERNUMBER']}
    )
    documentos.append(doc)

# 3. Creación de la Base Vectorial (Embeddings all-MiniLM-L6-v2)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=documentos, embedding=embeddings)

print("✓ ChromaDB creado correctamente.")

### **Sección 7: Configuración de la Cadena RAG y Chat Interactivo**

In [ ]:
# 1. Instalación y Preparación Integral del RAG
!pip install -q chromadb sentence-transformers langchain-huggingface

import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 2. Re-generación de la Base Vectorial (Asegurando existencia de 'vectorstore')
df_ventas = pd.read_csv('ventas.csv')
documentos = []
for _, row in df_ventas.iterrows():
    contenido = f"Orden: {row['ORDERNUMBER']} | Cliente: {row['CUSTOMERNAME']} | Producto: {row['PRODUCTLINE']} | Venta: {row['SALES']} | Ubicación: {row['CITY']}, {row['COUNTRY']}"
    documentos.append(Document(page_content=contenido, metadata={"order": row['ORDERNUMBER']}))

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=documentos, embedding=embeddings)
print("✓ ChromaDB inicializado correctamente.")

# 3. Configuración de la Cadena RAG
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

system_prompt = """
Eres un analista de datos experto en ventas. Responde en español de forma clara y concisa.
Contexto de las columnas:
- SALES: Monto total | STATUS: Estado | PRODUCTLINE: Línea de producto.

Contexto recuperado:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{question}")])
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Interfaz de Chat
print("--- CHAT ANALÍTICO ACTIVO (Escribe 'salir' para terminar) ---")
while True:
    try:
        query = input("Tú: ")
        if query.lower() in ['salir', 'exit', 'quit']:
            print("Asistente: ¡Hasta luego!")
            break
        respuesta = rag_chain.invoke(query)
        print(f"\nAsistente: {respuesta}\n¿Algo más?")
    except KeyboardInterrupt: break
    except Exception as e: print(f"Error: {e}")

In [21]:
import os
import pandas as pd
from google.colab import userdata

# 1. Instalación de dependencias críticas
print("Instalando dependencias...")
!pip install -q langchain langchain-community langchain-mistralai langchain-huggingface chromadb sentence-transformers langchain-experimental

from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 2. Inicialización de LLM y Datos
try:
    api_key = userdata.get('MISTRAL_API_KEY')
    llm = ChatMistralAI(api_key=api_key, model='mistral-small-latest')

    df_ventas = pd.read_csv('ventas.csv')
    print("✓ Datos cargados.")

    # 3. Re-creación de Base Vectorial
    documentos = [Document(page_content=f"Orden: {r['ORDERNUMBER']} | Cliente: {r['CUSTOMERNAME']} | Venta: {r['SALES']} | Pais: {r['COUNTRY']}") for _, r in df_ventas.iterrows()]
    embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    vectorstore = Chroma.from_documents(documents=documentos, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

    # 4. Cadena RAG
    prompt = ChatPromptTemplate.from_template("""Analista de ventas. Responde en español.
    Contexto: {context}
    Pregunta: {question}""")

    rag_chain = (
        {'context': retriever | (lambda docs: '\n\n'.join(d.page_content for d in docs)), 'question': RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )

    print("--- SISTEMA REESTABLECIDO: CHAT ACTIVO ---")
    while True:
        q = input("Tú: ")
        if q.lower() in ['salir', 'exit']: break
        print(f"\nAsistente: {rag_chain.invoke(q)}\n")

except Exception as e:
    print(f"❌ Error al reiniciar: {e}. Asegúrate de que MISTRAL_API_KEY esté en Secrets y ventas.csv exista.")

Instalando dependencias...
✓ Datos cargados.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

❌ Error al reiniciar: Could not import chromadb python package. Please install it with `pip install chromadb`.. Asegúrate de que MISTRAL_API_KEY esté en Secrets y ventas.csv exista.


In [20]:
from google.colab import userdata
from langchain_mistralai import ChatMistralAI

try:
    api_key = userdata.get('MISTRAL_API_KEY')
    llm_test = ChatMistralAI(api_key=api_key, model='mistral-small-latest')
    test_res = llm_test.invoke("Hola, ¿estás listo?")
    print(f"✅ ¡Conexión exitosa! El modelo responde: {test_res.content}")
    print("Ahora puedes volver a ejecutar la celda anterior para iniciar el Chat.")
except Exception as e:
    print(f"❌ Error persistente: {e}\n")
    print("Asegúrate de haber activado el interruptor de 'Notebook access' en el menú de Secrets (icono de llave).")

✅ ¡Conexión exitosa! El modelo responde: ¡Hola! Sí, estoy listo para ayudarte con lo que necesites. 😊 ¿En qué puedo asistirte hoy?
Ahora puedes volver a ejecutar la celda anterior para iniciar el Chat.


In [3]:
# 0. Corrección forzada de versiones y dependencias
!pip install -q --upgrade "opentelemetry-api>=1.21.0" "opentelemetry-sdk>=1.21.0" "chromadb>=0.4.15"

import pandas as pd
from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Preparación del Entorno
try:
    final_api_key = globals().get('api_key') or userdata.get('MISTRAL_API_KEY')
    llm = ChatMistralAI(api_key=final_api_key, model='mistral-small-latest')
    df_ventas = pd.read_csv('ventas.csv')
    print("✓ Entorno y datos cargados.")
except Exception as e:
    print(f"⚠️ Error al configurar el entorno: {e}")

# 2. Creación de Documentos y Vector Store (CON CAMPOS COMPLETOS)
print("Construyendo base de conocimientos vectorial detallada...")
documentos = []
for _, r in df_ventas.iterrows():
    # Incluimos explícitamente STATUS y CUSTOMERNAME para que sean recuperables
    texto = f"Orden: {r['ORDERNUMBER']} | Cliente: {r['CUSTOMERNAME']} | Producto: {r['PRODUCTLINE']} | Estado: {r['STATUS']} | Venta: {r['SALES']} | Ciudad: {r['CITY']} | Pais: {r['COUNTRY']}"
    documentos.append(Document(page_content=texto))

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = Chroma.from_documents(documents=documentos, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 10})

# 3. Definición de la Cadena RAG
prompt = ChatPromptTemplate.from_template("""Eres un analista experto.
IMPORTANTE: Solo tienes acceso a una MUESTRA de los datos (el contexto recuperado).
Si te preguntan por el TOTAL de filas o ventas del dataset completo, aclara que solo ves los registros más relevantes recuperados.

Contexto: {context}
Pregunta: {question}""")

rag_chain = (
    {'context': retriever | (lambda docs: '\n'.join(d.page_content for d in docs)), 'question': RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# 4. Interfaz de Usuario
print("\n--- ✅ SISTEMA RAG ACTUALIZADO (Con búsqueda por Cliente/Estado) ---")
while True:
    try:
        user_input = input("Pregunta: ")
        if user_input.lower() in ['salir', 'exit', 'quit']:
            print("Cerrando sesión. ¡Hasta pronto!")
            break
        print(f"\nAsistente: {rag_chain.invoke(user_input)}\n")
    except KeyboardInterrupt: break
    except Exception as e: print(f"Error: {e}")

✓ Entorno y datos cargados.
Construyendo base de conocimientos vectorial detallada...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- ✅ SISTEMA RAG ACTUALIZADO (Con búsqueda por Cliente/Estado) ---
Pregunta: cuando es el cumpleaños de uninorte

Asistente: No tengo acceso a información relacionada con el cumpleaños de **Uninorte** (Universidad del Norte en Colombia) en los datos proporcionados. Los registros que tengo disponibles solo incluyen datos de órdenes de compra con detalles como cliente, producto, estado, venta, ciudad y país, pero no hay información sobre fechas de cumpleaños o eventos institucionales.

Si necesitas saber la fecha de fundación o aniversario de Uninorte, te recomiendo consultar fuentes oficiales como su [página web](https://www.uninorte.edu.co) o sus redes sociales, donde suelen compartir este tipo de información.

Pregunta: Pregunta: Dime detalles de las órdenes que están en estado 'On Hold'.  Asistente: **Respuesta:**  No tengo acceso a información sobre órdenes en estado **"On Hold"** en los datos proporcionados. Solo puedo ver registros de órdenes **finalizadas** (con ventas registra